In [0]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Imports

In [0]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import os

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries loaded successfully")

OUTPUT_PATH_figures = "/Users/chamika/Desktop/git/machine_learning/Question_1/outputs/figures/"
DATA_PATH_raw    = "/Users/chamika/Desktop/git/machine_learning/Question_1/data/raw/"
DATA_PATH_processed    = "/Users/chamika/Desktop/git/machine_learning/Question_1/data/processed/"

# Load the Cleaned Data

In [0]:
# Load cleaned data (output of ETL notebook)
df_clean = pd.read_csv(DATA_PATH_processed + 'df_clean.csv')
print(f"✅ Loaded df_clean: {df_clean.shape}")

# Feature Engineering 

In [0]:
# Cell 19: Feature Engineering — Creating 6 new meaningful features

print("=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

# ── Feature 1 & 2: Temporal features from TransactionDT ──────
# TransactionDT is seconds elapsed since a reference point
df_clean['hour'] = (df_clean['TransactionDT'] / 3600) % 24
df_clean['hour'] = df_clean['hour'].astype(int)

df_clean['day_of_week'] = (df_clean['TransactionDT'] / (3600 * 24)) % 7
df_clean['day_of_week'] = df_clean['day_of_week'].astype(int)

print("\n✅ Feature 1: 'hour' — Hour of day (0–23)")
print(f"   Unique values: {sorted(df_clean['hour'].unique())}")

print("\n✅ Feature 2: 'day_of_week' — Day of week (0–6)")
print(f"   Unique values: {sorted(df_clean['day_of_week'].unique())}")

# ── Feature 3: Log-transformed transaction amount ────────────
df_clean['amt_log'] = np.log1p(df_clean['TransactionAmt'])

print("\n✅ Feature 3: 'amt_log' — log(TransactionAmt + 1)")
print(f"   Mean: {df_clean['amt_log'].mean():.4f}")
print(f"   Std:  {df_clean['amt_log'].std():.4f}")

In [0]:
# Clean up diagnostic test columns — keep only the final card3-based features
card_cols_to_test = ['card1', 'card2', 'card3', 'card5']  # re-declared for reproducibility

test_cols_to_drop = []
for card_col in card_cols_to_test:
    test_cols_to_drop.append(f'{card_col}_avg_amt')
    test_cols_to_drop.append(f'amt_vs_{card_col}_avg')

test_cols_to_drop = [c for c in test_cols_to_drop if c in df_clean.columns]
df_clean = df_clean.drop(columns=test_cols_to_drop)

print(f"Dropped {len(test_cols_to_drop)} diagnostic test columns: {test_cols_to_drop}")
print(f"Shape after cleanup: {df_clean.shape}")

# ── Feature 4: Card-level average transaction amount (using card3) ───
# card3 selected over card1/card2/card5 based on empirical correlation
# with isFraud (card3: r=0.0535 vs card1: r=0.0324, card2: r=0.0387, card5: r=0.0217)
card_avg = df_clean.groupby('card3')['TransactionAmt'].transform('mean')
df_clean['card_avg_amt'] = card_avg

print("\n✅ Feature 4: 'card_avg_amt' — Mean TransactionAmt per card3")
print(f"   Mean: {df_clean['card_avg_amt'].mean():.4f}")
print(f"   Std:  {df_clean['card_avg_amt'].std():.4f}")

# ── Feature 5: Transaction amount vs card3 average ────────────
df_clean['amt_vs_card_avg'] = (
    df_clean['TransactionAmt'] / (df_clean['card_avg_amt'] + 1e-9)
)

print("\n✅ Feature 5: 'amt_vs_card_avg' — TransactionAmt / card_avg_amt (grouped by card3)")
print(f"   Mean: {df_clean['amt_vs_card_avg'].mean():.4f}")
print(f"   Std:  {df_clean['amt_vs_card_avg'].std():.4f}")
print(f"   Correlation with isFraud: {df_clean['amt_vs_card_avg'].corr(df_clean['isFraud']):.4f}")

### Feature Selection: Card-Level Spending Deviation

The engineered feature amt_vs_card_avg (transaction amount relative to a card's average spend) requires choosing which of the dataset's card identifier columns to group by. The dataset contains six card-related columns: card1, card2, card3, card5 (numerical, higher-cardinality identifiers) and card4, card6 (categorical, only 4 unique values each — card network and card type respectively). The categorical pair was excluded from consideration, as grouping by only 4 categories would produce an average so coarse (spanning hundreds of thousands of transactions per group) that it would closely resemble the global average rather than capturing card-specific behaviour.
To select the most informative grouping column among the four numerical candidates, each was tested by computing TransactionAmt relative to its group average and measuring correlation with isFraud:
| Grouping column | Correlation with isFraud |
|---|---|
| `card1` | 0.0324 |
| `card2` | 0.0387 |
| `card3` | **0.0535** |
| `card5` | 0.0217 |

card3 showed the strongest correlation — approximately 65% higher than card1, which had been the initial default choice. card3 was therefore selected as the grouping variable for the final card_avg_amt and amt_vs_card_avg features, rather than retaining all four as separate (largely redundant) features feeding into the later PCA/feature-selection step.
It is worth noting that all four correlations are individually modest in absolute terms — considerably weaker than the dataset's strongest univariate predictors (e.g. the top V-columns, r≈0.38). This is not necessarily disqualifying: the value of amt_vs_card_avg as a behavioural-deviation signal may lie more in its interaction with other features within a tree-based model (e.g. XGBoost) than in its standalone linear correlation with the target.


In [0]:
# ── Feature 6 (corrected): Email domain match flag ───────────────────────
# Computed from the ORIGINAL raw string values in `df` (never Label Encoded),
# rather than the already-encoded df_clean columns — avoids the mismatched
# integer-code problem, since each LabelEncoder was fit independently per column.

# ── Feature 6 (corrected): Email domain match flag ───────────────────────
email_match_raw = (df_clean['P_emaildomain_raw'] == df_clean['R_emaildomain_raw']) & \
                   df_clean['P_emaildomain_raw'].notna() & df_clean['R_emaildomain_raw'].notna()

df_clean['email_match'] = email_match_raw.astype(int).values
df_clean = df_clean.drop(columns=['P_emaildomain_raw', 'R_emaildomain_raw'])  # cleanup — RF importance can't handle strings

print("\n✅ Feature 6 (corrected): 'email_match' — computed on raw domain strings")
print(f"   Match (1):    {df_clean['email_match'].sum():,} transactions")
print(f"   No match (0): {(df_clean['email_match']==0).sum():,} transactions")
print(f"\n📊 Final shape after feature engineering: {df_clean.shape}")

### Feature 6: Email Domain Match (email_match)
This feature checks whether the purchaser's email domain (P_emaildomain) and the recipient's email domain (R_emaildomain) are the same, on the hypothesis that a mismatch could indicate fraud (e.g. account takeover, or a case where the purchaser is not the true account owner).
Both columns (P_emaildomain, R_emaildomain) had already been Label Encoded independently, so identical domains (e.g. "gmail.com" in both) were not guaranteed to receive the same numeric code, making a direct comparison unreliable. This was corrected by computing the match using the original, unencoded domain strings (preserved in the raw dataframe) instead of the encoded values.

| P_emaildomain | R_emaildomain | P == R | P.notna() | R.notna() | Final `email_match` |
|---|---|---|---|---|---|
| "gmail.com" | "gmail.com" | True | True | True | 1 (genuine match) |
| "gmail.com" | "yahoo.com" | False | True | True | 0 (genuine mismatch) |
| "gmail.com" | NaN | False* | True | False | 0 (treated as no match) |
| NaN | "gmail.com" | False* | False | True | 0 (treated as no match) |
| NaN | NaN | False* | False | False | 0 (treated as no match) |

*In pandas, comparing `NaN` to anything (including `NaN == NaN`) always evaluates to `False`.

# Validate Features Against Target

In [0]:
# Cell 20: Validate new features — check fraud rate across each

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

new_features = ['hour', 'day_of_week', 'amt_log', 
                'card_avg_amt', 'amt_vs_card_avg', 'email_match']

for idx, feat in enumerate(new_features):
    
    if feat in ['hour', 'day_of_week', 'email_match']:
        # Categorical-style: fraud rate per value
        fraud_rate = df_clean.groupby(feat)['isFraud'].mean() * 100
        axes[idx].bar(fraud_rate.index, fraud_rate.values,
                      color='#e74c3c', alpha=0.7, edgecolor='black', 
                      linewidth=0.5)
        axes[idx].axhline(y=df_clean['isFraud'].mean() * 100,
                          color='blue', linestyle='--', 
                          linewidth=1.5, label='Overall avg')
        axes[idx].set_xlabel(feat)
        axes[idx].set_ylabel('Fraud Rate (%)')
        axes[idx].legend(fontsize=8)
        
    else:
        # Continuous: distribution by class
        legit = df_clean[df_clean['isFraud']==0][feat]
        fraud = df_clean[df_clean['isFraud']==1][feat]
        axes[idx].hist(legit, bins=50, alpha=0.5, color='#2ecc71',
                       label='Legitimate', density=True)
        axes[idx].hist(fraud, bins=50, alpha=0.5, color='#e74c3c',
                       label='Fraud', density=True)
        axes[idx].set_xlabel(feat)
        axes[idx].set_ylabel('Density')
        axes[idx].legend(fontsize=8)

    axes[idx].set_title(f'Feature: {feat}', fontsize=12, fontweight='bold')

plt.suptitle('New Engineered Features — Fraud Signal Validation',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'engineered_features.png', 
            dpi=150, bbox_inches='tight')
plt.show()

# Print fraud rate stats per feature
print("📊 Fraud rate by email_match:")
print(df_clean.groupby('email_match')['isFraud'].mean().round(4) * 100)

print("\n📊 Fraud rate by hour (top 5 riskiest hours):")
print((df_clean.groupby('hour')['isFraud'].mean() * 100)
      .sort_values(ascending=False).head(5).round(2))

print("\n📊 Fraud rate by day_of_week:")
print((df_clean.groupby('day_of_week')['isFraud'].mean() * 100).round(2))

# Feature Selection using Random Forest Importance

In [0]:
# Cell 21: Feature selection — Random Forest importance

print("Running feature importance (this may take 2–3 minutes)...")

# Use a sample for speed
sample_df = df_clean.sample(n=50000, random_state=42)

X_sample = sample_df.drop(columns=['TransactionID', 'isFraud'])
y_sample = sample_df['isFraud']

# Quick RF for importance
rf_selector = RandomForestClassifier(
    n_estimators=50, 
    max_depth=8,
    random_state=42, 
    n_jobs=-1,
    class_weight='balanced'
)
rf_selector.fit(X_sample, y_sample)

# Get importance
importance_df = pd.DataFrame({
    'Feature': X_sample.columns,
    'Importance': rf_selector.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\n✅ Feature importance calculated")
print(f"\nTop 30 most important features:")
print(importance_df.head(30).to_string(index=False))

# Plot top 30
fig, ax = plt.subplots(figsize=(12, 10))
top30 = importance_df.head(30)

colors = ['#e74c3c' if f in new_features else '#3498db' 
          for f in top30['Feature']]

ax.barh(top30['Feature'][::-1], top30['Importance'][::-1], 
        color=colors[::-1], edgecolor='black', linewidth=0.5)
ax.set_title('Top 30 Features by Random Forest Importance\n'
             '(Red = Engineered features)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Feature Importance Score')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='Engineered features'),
                   Patch(facecolor='#3498db', label='Original features')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'feature_importance.png', 
            dpi=150, bbox_inches='tight')
plt.show()

# Select Final Feature Set & Save

In [0]:
# Cell 22: Select final features and save

# Keep top 100 features + our engineered features + target
top_features = importance_df.head(100)['Feature'].tolist()

# Make sure all engineered features are included
for f in new_features:
    if f not in top_features:
        top_features.append(f)

# Final columns
final_cols = ['TransactionID'] + top_features + ['isFraud']
df_final = df_clean[final_cols].copy()

print("=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)
print(f"Original shape:      {df_clean.shape}")
print(f"Final shape:         {df_final.shape}")
print(f"Features selected:   {len(top_features)}")
print(f"Engineered features: {new_features}")
print(f"Target:              isFraud")

# Save final dataset
df_final.to_csv(DATA_PATH_processed + 'df_final.csv', index=False)
print(f"\n✅ Final dataset saved to:")
print(f"   {DATA_PATH_processed}df_final.csv")

# Final class balance check
print(f"\n📊 Final target distribution:")
print(f"   Legitimate: {(df_final['isFraud']==0).sum():,} ({(df_final['isFraud']==0).mean()*100:.2f}%)")
print(f"   Fraud:      {(df_final['isFraud']==1).sum():,} ({(df_final['isFraud']==1).mean()*100:.2f}%)")

# PCA Analysis

In [0]:
# Cell 23: PCA — Dimensionality Reduction Analysis
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("Running PCA analysis...")

# Use df_final (already feature-selected to 103 features)
X_pca = df_final.drop(columns=['TransactionID', 'isFraud'])
y_pca = df_final['isFraud']

# Sample for speed
sample_idx = X_pca.sample(n=50000, random_state=42).index
X_sample_pca = X_pca.loc[sample_idx]
y_sample_pca = y_pca.loc[sample_idx]

# Step 1: Standardise (PCA requires scaled data)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sample_pca)

# Step 2: Fit PCA with all components
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

# Step 3: Calculate cumulative variance
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = np.argmax(cumvar >= 0.95) + 1
n_90 = np.argmax(cumvar >= 0.90) + 1
n_80 = np.argmax(cumvar >= 0.80) + 1

print(f"\n📊 PCA Variance Summary:")
print(f"   Components for 80% variance: {n_80}")
print(f"   Components for 90% variance: {n_90}")
print(f"   Components for 95% variance: {n_95}")
print(f"   Total original features:     {X_scaled.shape[1]}")
print(f"   Dimensionality reduction:    {X_scaled.shape[1]} → {n_95} "
      f"({(1 - n_95/X_scaled.shape[1])*100:.1f}% reduction)")

# ── Plot 1 & 2: Variance explained ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cumulative variance plot
axes[0].plot(range(1, len(cumvar)+1), cumvar * 100,
             color='#3498db', linewidth=2)
axes[0].axhline(y=95, color='red', linestyle='--', 
                linewidth=1.5, label='95% threshold')
axes[0].axhline(y=90, color='orange', linestyle='--', 
                linewidth=1.5, label='90% threshold')
axes[0].axhline(y=80, color='green', linestyle='--', 
                linewidth=1.5, label='80% threshold')
axes[0].axvline(x=n_95, color='red', linestyle=':', linewidth=1.5)
axes[0].axvline(x=n_90, color='orange', linestyle=':', linewidth=1.5)
axes[0].axvline(x=n_80, color='green', linestyle=':', linewidth=1.5)
axes[0].fill_between(range(1, len(cumvar)+1), cumvar * 100,
                      alpha=0.1, color='#3498db')
axes[0].set_title('Cumulative Explained Variance by PCA Components',
                   fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Components')
axes[0].set_ylabel('Cumulative Variance Explained (%)')
axes[0].legend(fontsize=9)
axes[0].set_xlim(1, len(cumvar))
axes[0].set_ylim(0, 101)

# Annotate thresholds
axes[0].annotate(f'{n_80} components\n(80%)', 
                  xy=(n_80, 80), xytext=(n_80+5, 70),
                  fontsize=8, color='green',
                  arrowprops=dict(arrowstyle='->', color='green'))
axes[0].annotate(f'{n_95} components\n(95%)', 
                  xy=(n_95, 95), xytext=(n_95+5, 85),
                  fontsize=8, color='red',
                  arrowprops=dict(arrowstyle='->', color='red'))

# Individual variance per component (top 30)
axes[1].bar(range(1, 31),
            pca_full.explained_variance_ratio_[:30] * 100,
            color='#3498db', edgecolor='black', linewidth=0.5)
axes[1].set_title('Variance Explained by First 30 Components',
                   fontsize=13, fontweight='bold')
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Variance Explained (%)')

plt.suptitle('PCA — Dimensionality Reduction Analysis',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'pca_variance.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Plot 3: 2D PCA scatter — fraud vs legitimate ─────────────
pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 7))

# Plot legitimate first (background)
legit_mask = y_sample_pca == 0
fraud_mask = y_sample_pca == 1

ax.scatter(X_2d[legit_mask, 0], X_2d[legit_mask, 1],
           c='#2ecc71', alpha=0.2, s=3, label='Legitimate')
ax.scatter(X_2d[fraud_mask, 0], X_2d[fraud_mask, 1],
           c='#e74c3c', alpha=0.5, s=5, label='Fraud')

ax.set_title('PCA 2D Projection — Fraud vs Legitimate Transactions',
             fontsize=14, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(markerscale=3, fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'pca_2d_scatter.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ PCA Analysis Complete")
print(f"   Decision: Use top {n_95} components to retain 95% variance")
print(f"   This reduces dimensionality by "
      f"{(1 - n_95/X_scaled.shape[1])*100:.1f}% for model training")